# 02 · Silver layer: cleaned and standardized data



**Purpose.** Clean and standardize the Bronze data with Spark DataFrame transformations.

**Inputs.** `workspace.car_insurance.claims_bronze` (name in `BRONZE_TABLE`).

**Outputs.** `workspace.car_insurance.claims_silver` (name in `SILVER_TABLE`).

**Assumptions.** 
- A column named `_corrupt_record` was created in the Bronze Layer

**What the next notebook relies on (the table contract)**

- The 19 source columns, same names, one row per driver. `ID` must stay unique and non-null: it joins drivers to their Phase 2 split.
- Numeric columns stored as numbers and category columns as text. Keep `POSTAL_CODE` as text.
- **Do not impute, scale or encode here.** Phase 2 fitted those steps on training drivers only; doing them on all rows leaks test data. They happen inside each model's Spark ML pipeline.
- Every removed row changes the test set. Phase 2 kept all 10,000 drivers, so report any rule that removes rows and its effect.

**Shared helpers.** The first code cell runs `00_setup`, which defines the table names (`BRONZE_TABLE`, `SILVER_TABLE`, ...), the file paths (`CLAIMS_CSV`), and `write_delta` / `validate_tables`. Use those names instead of typing table names, so everyone writes to the same place.

In [0]:
%run ./00_setup

# 00 · Shared setup: storage, table names and Delta helpers

**Purpose.** One place for everything the other notebooks share: where the data lives, what each Delta table is called, how tables are written and verified, and how models are split and scored. Keeping it here means all team members write to the same tables in the same way.

**How it is used.** This notebook is not run on its own. Every other notebook starts with a cell containing only `%run ./00_setup`, which runs these cells and makes the names and functions below available.

**Inputs.** None.

**Outputs.** No tables. It creates the schema `workspace.car_insurance` and the volumes `raw` (source files) and `models` (saved Spark ML models) if they do not exist yet.

**Assumptions**

- The catalog is named `workspace` (the Databricks Free Edition default). If yours differs, change `CATALOG` below; nothing else needs editing.
- Serverless compute.

**What it provides**

| Name | Use |
|---|---|
| `BRONZE_TABLE`, `SILVER_TABLE`, `GOLD_FEATURES_TABLE`, `GOLD_SUMMARY_TABLE`, `SPLIT_TABLE` | Full table names. Always use these instead of typing a table name |
| `CLAIMS_CSV`, `SPLIT_CSV`, `MODEL_DIR` | Paths inside the volumes |
| `require_source_files(...)` | Stops with a clear message if an uploaded file is missing |
| `write_delta(df, table_name)` | Saves a DataFrame as a Delta table |
| `validate_tables({...})` | Reads tables back and checks format, rows, columns and key |
| `load_model_splits()` | Train, validation and test DataFrames with the Phase 2 split |
| `evaluate_classifier(predictions)` | Accuracy, precision, recall, F1, ROC-AUC and confusion counts |

Setup loaded: tables in workspace.car_insurance, files in /Volumes/workspace/car_insurance


In [0]:
from pyspark.sql import functions as F

bronze = spark.table(BRONZE_TABLE)
bronze_rows = bronze.count()

## 1. Inspect and clean

The inspection cell below shows what Bronze contains before any rule is applied: malformed records, duplicates, missing values per column, and every value of the category and 0/1 columns. The cleaning rules are based on it:

| Rule | Why |
|---|---|
| Drop rows flagged in `_corrupt_record` | A line Spark could not parse cannot be trusted |
| Trim spaces; empty text becomes missing | Consistent values; a blank field is missing data |
| Lowercase the category columns | One spelling per category |
| Group `DRIVING_EXPERIENCE`: 0-9y → Beginner, 10-19y → Advanced, 20-29y and 30y+ → Expert | The grouping used in Phases 1 and 2; done here because it merges two original categories |
| 0/1 flags to integer only if 0 or 1; counts to integer only if a whole number ≥ 0; `CREDIT_SCORE` and `ANNUAL_MILEAGE` to double | Correct types; an impossible value becomes missing instead of a wrong number |
| Drop rows with a missing `ID` or `OUTCOME` | `ID` joins the Phase 2 split and `OUTCOME` is the target |
| Drop exact duplicate rows; stop if an `ID` appears twice | One row per driver |

Missing `CREDIT_SCORE` and `ANNUAL_MILEAGE` are not filled here, because the models fill them from training drivers only. Drivers whose values are identical apart from `ID` are different drivers and are kept. The quality check at the end of this section shows the effect of the rules.

In [0]:
# Inspect Bronze before cleaning: malformed records, duplicates, missing values and category values.
from functools import reduce

INSPECT_COLUMNS = ["AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE", "EDUCATION", "INCOME", "VEHICLE_YEAR",
                   "VEHICLE_TYPE", "POSTAL_CODE", "VEHICLE_OWNERSHIP", "MARRIED", "CHILDREN", LABEL_COLUMN]

print(f"Bronze rows: {bronze_rows:,}")
print(f"Malformed records (_corrupt_record): {bronze.filter(F.col('_corrupt_record').isNotNull()).count():,}")
print(f"Duplicate IDs: {bronze_rows - bronze.select(ID_COLUMN).distinct().count():,}")
print(f"Exact duplicate rows: {bronze_rows - bronze.select(*SOURCE_COLUMNS).distinct().count():,}")
print(f"Rows identical apart from ID: "
      f"{bronze_rows - bronze.select(*[c for c in SOURCE_COLUMNS if c != ID_COLUMN]).distinct().count():,}")

# Missing values per column (null or blank text)
display(bronze.select([F.sum((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")).alias(c)
                       for c in SOURCE_COLUMNS]))

# Every value of the category and 0/1 columns, to spot misspellings or inconsistent labels
display(reduce(lambda a, b: a.unionByName(b), [
    bronze.groupBy(F.col(c).alias("value")).count().select(F.lit(c).alias("column"), "value", "count")
    for c in INSPECT_COLUMNS]).orderBy("column", "value"))

Bronze rows: 10,000
Malformed records (_corrupt_record): 0
Duplicate IDs: 0
Exact duplicate rows: 0
Rows identical apart from ID: 12


ID,AGE,GENDER,RACE,DRIVING_EXPERIENCE,EDUCATION,INCOME,CREDIT_SCORE,VEHICLE_OWNERSHIP,VEHICLE_YEAR,MARRIED,CHILDREN,POSTAL_CODE,ANNUAL_MILEAGE,VEHICLE_TYPE,SPEEDING_VIOLATIONS,DUIS,PAST_ACCIDENTS,OUTCOME
0,0,0,0,0,0,0,982,0,0,0,0,0,957,0,0,0,0,0


column,value,count
AGE,16-25,2016
AGE,26-39,3063
AGE,40-64,2931
AGE,65+,1990
CHILDREN,0.0,3112
CHILDREN,1.0,6888
DRIVING_EXPERIENCE,0-9y,3530
DRIVING_EXPERIENCE,10-19y,3299
DRIVING_EXPERIENCE,20-29y,2119
DRIVING_EXPERIENCE,30y+,1052


### Parse rows

In [0]:
parsed_rows = bronze.filter(F.col("_corrupt_record").isNull())
parsed_counts = parsed_rows.count()
clean = parsed_rows.select(*SOURCE_COLUMNS)

### Ensure data types are correct

In [0]:
for name in SOURCE_COLUMNS:
    value = F.trim(F.col(name))
    clean = clean.withColumn(name, F.when(value != "", value))

categorical_columns = [
    "AGE", "GENDER", "RACE", "DRIVING_EXPERIENCE",
    "EDUCATION", "INCOME", "VEHICLE_YEAR", "VEHICLE_TYPE"
]

for name in categorical_columns:
    clean = clean.withColumn(name, F.lower(F.col(name)))

#### Relabel DRIVING_EXPERIENCE

In [0]:
# shoud be done here because we are regrouping two groups
experience = F.col("DRIVING_EXPERIENCE")
clean = clean.withColumn(
    "DRIVING_EXPERIENCE",
    F.when(experience == "0-9y", "Beginner")
    .when(experience == "10-19y", "Advanced")
    .when(experience.isin("20-29y", "30y+"), "Expert")
    .otherwise(experience)
)

#### Conver to int and doubles

In [0]:
for name in ["MARRIED", "CHILDREN", "VEHICLE_OWNERSHIP", LABEL_COLUMN]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    clean = clean.withColumn(name, F.when(value.isin(0.0, 1.0), value.cast("int")))

for name in ["SPEEDING_VIOLATIONS", "DUIS", "PAST_ACCIDENTS"]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    valid_count = value.between(0, 2147483647) & ((value%1) == 0)
    clean = clean.withColumn(name, F.when(valid_count, value.cast("int")))

for name in ["CREDIT_SCORE", "ANNUAL_MILEAGE"]:
    value = F.expr(f"try_cast(`{name}` AS DOUBLE)")
    clean = clean.withColumn(name, F.when(F.abs(value) < float("inf"), value))

### Validate values and remove duplicates

In [0]:
valid = clean.filter(
    F.col(ID_COLUMN).isNotNull() & F.col(LABEL_COLUMN).isNotNull()
)
valid_count = valid.count()

In [0]:
silver = valid.dropDuplicates()
silver_count = silver.count()

In [0]:
id_conflicts = silver.groupBy(ID_COLUMN).count().filter(F.col("count") > 1)

if id_conflicts.limit(1).count() > 0:
    display(id_conflicts.limit(20))
    raise ValueError("ID conflicts must be addressed")

### Quality check after cleaning

Missing values per column before and after cleaning, with the Silver type of each column. A value that a rule rejected (for example a 0/1 flag holding 2) would show up as an increase in `set_missing_by_rules`.

In [0]:
def missing_counts(df):
    """Missing values per source column: null, or blank once converted to text."""
    return df.select([F.sum((F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).cast("int")).alias(c)
                      for c in SOURCE_COLUMNS]).first().asDict()

missing_before, missing_after = missing_counts(bronze), missing_counts(silver)
silver_types = dict(silver.dtypes)
display(spark.createDataFrame(
    [(c, silver_types[c], int(missing_before[c] or 0), int(missing_after[c] or 0),
      int(missing_after[c] or 0) - int(missing_before[c] or 0)) for c in SOURCE_COLUMNS],
    "column string, silver_type string, missing_in_bronze long, missing_in_silver long, set_missing_by_rules long"))

column,silver_type,missing_in_bronze,missing_in_silver,set_missing_by_rules
ID,string,0,0,0
AGE,string,0,0,0
GENDER,string,0,0,0
RACE,string,0,0,0
DRIVING_EXPERIENCE,string,0,0,0
EDUCATION,string,0,0,0
INCOME,string,0,0,0
CREDIT_SCORE,double,982,982,0
VEHICLE_OWNERSHIP,int,0,0,0
VEHICLE_YEAR,string,0,0,0


## 2. Write and verify the Delta table

In [0]:
print(f"Bronze rows: {bronze.count()}")
print(f"Flagged CSV rows removed: {bronze.count() - parsed_counts}")
print(f"Rows removed for missing ID or invalid outcome: {parsed_counts - valid_count}")
print(f"Duplicate standardized rows removed: {valid_count - silver_count}")
print(f"Silver rows: {silver_count}")

Bronze rows: 10000
Flagged CSV rows removed: 0
Rows removed for missing ID or invalid outcome: 0
Duplicate standardized rows removed: 0
Silver rows: 10000


In [0]:
expected_silver_rows = silver.count()

write_delta(silver, SILVER_TABLE)

# Reads the table back from the catalog: format, row count, columns, and ID uniqueness.
validate_tables({SILVER_TABLE: expected_silver_rows})

Wrote workspace.car_insurance.claims_silver


table,format,expected_rows,actual_rows,columns,status,problems
workspace.car_insurance.claims_silver,delta,10000,10000,19,PASS,
